# O modelo

O XGBoost, no contexto do HMS, atua como um modelo de árvores de decisão impulsionadas por gradiente que utiliza características extraídas do EEG nos domínios do tempo e da frequência. Ele trabalha com vetores de alta dimensão contendo estatísticas, potências espectrais e correlações entre canais. Ao otimizar uma função de perda, o modelo aprende relações não lineares e gera distribuições probabilísticas capazes de identificar padrões complexos, mantendo boa eficiência computacional e robustez a ruídos.

Leitura dos dados já separados e tratados anteriormente

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# Criando as pastas de destino no ambiente local do Colab
!mkdir -p /content/eeg_data
!mkdir -p /content/spectrogram_data

# Descompactando os arquivos
print("Extraindo EEGs...")
!unzip -q "/content/drive/MyDrive/dados hms/eeg_processado_mvp.zip" -d /content/eeg_data

print("Extraindo Espectrogramas...")
!unzip -q "/content/drive/MyDrive/dados hms/spectrogram_processado_mvp.zip" -d /content/spectrogram_data

print("Extração concluída!")

Extraindo EEGs...
Extraindo Espectrogramas...
Extração concluída!


In [3]:
import pandas as pd
import numpy as np

# Carregando os metadados
train_df = pd.read_csv("/content/drive/MyDrive/dados hms/train_final.csv")
test_df = pd.read_csv("/content/drive/MyDrive/dados hms/test_final.csv")

print(f"Treino: {train_df.shape[0]} linhas")
print(f"Teste: {test_df.shape[0]} linhas")

example_id = train_df.iloc[0]['spectrogram_id']


Treino: 13671 linhas
Teste: 3418 linhas


In [4]:
!ls /content/spectrogram_data


example_id = train_df.iloc[0]['spectrogram_id']
file_path = f"/content/spectrogram_data/spectrogram_processado_mvp/{example_id}.npy"

data = np.load(file_path)
print("Shape do dado carregado:", data.shape)

spectrogram_processado_mvp
Shape do dado carregado: (300, 400)


1. Função de Extração de Features

In [5]:

from tqdm import tqdm

def extract_features(path, df):
    # Lista para armazenar as novas colunas
    all_features = []

    print("Extraindo features dos espectrogramas...")
    for j, row in tqdm(df.iterrows(), total=len(df)):
        # 1. Carregar o arquivo .npy
        file_path = f"{path}/{row['spectrogram_id']}.npy"
        img = np.load(file_path) # Shape (300, 400)

        # 2. Criar estatísticas básicas (Média e Desvio Padrão)
        mean_features = np.nanmean(img, axis=0)
        std_features = np.nanstd(img, axis=0)

        # 3. Combinar tudo em um único vetor para esta linha
        feature_vector = np.concatenate([mean_features, std_features])
        all_features.append(feature_vector)

    # Converter para DataFrame
    return np.array(all_features)

# Definindo os caminhos
PATH_IMG = "/content/spectrogram_data/spectrogram_processado_mvp"

# Executando a extração para Treino e Teste
X_train = extract_features(PATH_IMG, train_df)
X_test = extract_features(PATH_IMG, test_df)

print(f"\nShape final do X_train: {X_train.shape}")

Extraindo features dos espectrogramas...


100%|██████████| 13671/13671 [00:33<00:00, 405.45it/s]


Extraindo features dos espectrogramas...


100%|██████████| 3418/3418 [00:08<00:00, 404.60it/s]


Shape final do X_train: (13671, 800)


2. Preparação dos Alvos (Labels)

In [6]:
# Lista das colunas de alvos
target_cols = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']

# Criando o Y (Target)
y_train = train_df[target_cols].values

print(f"Shape do y_train: {y_train.shape}")

Shape do y_train: (13671, 6)


3.Preparação dos grupos

In [7]:
from sklearn.model_selection import GroupKFold
import xgboost as xgb

# Definindo o número de folds
# 5 folds
gkf = GroupKFold(n_splits=5)
train_df['fold'] = -1

for i, (train_idx, val_idx) in enumerate(gkf.split(train_df, y_train, groups=train_df['patient_id'])):
    train_df.loc[val_idx, 'fold'] = i

Configuração e Treino do XGBoost

In [9]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import log_loss

# 1. Configuração dos Parâmetros
xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 6,
    'learning_rate': 0.05,
    'max_depth': 6,
    'n_estimators': 1000,
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': 42,
    'early_stopping_rounds': 50
}

# 2. Definição do Fold de Validação
fold = 0
train_idx = train_df[train_df['fold'] != fold].index
val_idx = train_df[train_df['fold'] == fold].index

# 3. Preparação das matrizes
X_train_fold = X_train[train_idx].astype('float32')
y_train_fold = y_train[train_idx]
X_val_fold = X_train[val_idx].astype('float32')
y_val_fold = y_train[val_idx]

y_train_labels = np.argmax(y_train_fold, axis=1)
y_val_labels = np.argmax(y_val_fold, axis=1)

# 4. Inicialização do Modelo
model = xgb.XGBClassifier(**xgb_params)

# 5. Treinamento
print(f"Iniciando treinamento do Fold {fold} na GPU...")
model.fit(
    X_train_fold, y_train_labels,
    eval_set=[(X_val_fold, y_val_labels)],
    verbose=50
)

# 6. Predição de Probabilidades e Avaliação Final
preds = model.predict_proba(X_val_fold)

# Cálculo da Métrica da Competição (KL Divergence / Log Loss)
def kl_divergence(y_true, y_pred):
    # Pequena constante para evitar log(0)
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    y_true = np.clip(y_true, eps, 1 - eps)
    return np.mean(np.sum(y_true * np.log(y_true / y_pred), axis=1))

# Cálculo da Métrica
score = kl_divergence(y_val_fold, preds)

print("\n" + "="*30)
print(f"RESULTADO FINAL FOLD {fold}")
print(f"KL Divergence Score (Manual): {score:.4f}")
print("="*30)

Iniciando treinamento do Fold 0 na GPU...
[0]	validation_0-mlogloss:1.54619
[50]	validation_0-mlogloss:1.31160
[100]	validation_0-mlogloss:1.28354
[150]	validation_0-mlogloss:1.27923
[199]	validation_0-mlogloss:1.27925

RESULTADO FINAL FOLD 0
KL Divergence Score (Manual): 1.0372


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [13:11:06] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Ciclo de 5-Folds e Média de Predições

In [10]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import log_loss

# 1. Configuração de Parâmetros
xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 6,
    'learning_rate': 0.05,
    'max_depth': 6,
    'n_estimators': 1000,
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': 42,
    'early_stopping_rounds': 50
}

# 2. Configuração da Validação Cruzada
gkf = GroupKFold(n_splits=5)
all_oof_preds = np.zeros(y_train.shape)
all_test_preds = np.zeros((len(test_df), 6))
scores = []

# 3. Loop de Treino nos 5 Folds
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, y_train, groups=train_df['patient_id'])):
    print(f"\n" + "="*50)
    print(f" TREINANDO FOLD {fold} ")
    print("="*50)

    # Preparação dos dados do Fold
    X_tr, y_tr = X_train[train_idx].astype('float32'), y_train[train_idx]
    X_va, y_va = X_train[val_idx].astype('float32'), y_train[val_idx]

    y_tr_labels = np.argmax(y_tr, axis=1)
    y_va_labels = np.argmax(y_va, axis=1)

    # Inicialização e Treino
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(
        X_tr, y_tr_labels,
        eval_set=[(X_va, y_va_labels)],
        verbose=100
    )

    # Predições de Validação
    oof_fold_preds = model.predict_proba(X_va)
    all_oof_preds[val_idx] = oof_fold_preds

    # Acumular Predições de Teste (Média Simples)
    all_test_preds += model.predict_proba(X_test.astype('float32')) / 5.0

    # Cálculo do Score do Fold
    fold_score = kl_divergence(y_va, oof_fold_preds)
    scores.append(fold_score)
    print(f"Score Fold {fold}: {fold_score:.4f}")

    #Salvar o modelo do fold
    model.save_model(f'xgboost_fold_{fold}.json')

# 4. Resultados Finais
overall_score = np.mean(scores)
print(f"\n\nCV SCORE GLOBAL (Média dos Folds): {overall_score:.4f}")

submission = pd.DataFrame({'eeg_id': test_df['eeg_id'].values})
target_cols = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
submission[target_cols] = all_test_preds
submission.to_csv('submission_ensemble.csv', index=False)




 TREINANDO FOLD 0 
[0]	validation_0-mlogloss:1.54619
[100]	validation_0-mlogloss:1.28354
[199]	validation_0-mlogloss:1.27925
Score Fold 0: 1.0372

 TREINANDO FOLD 1 
[0]	validation_0-mlogloss:1.60305
[100]	validation_0-mlogloss:1.37784
[156]	validation_0-mlogloss:1.38182
Score Fold 1: 1.0914

 TREINANDO FOLD 2 
[0]	validation_0-mlogloss:1.58108
[100]	validation_0-mlogloss:1.23624
[186]	validation_0-mlogloss:1.24356
Score Fold 2: 0.9824

 TREINANDO FOLD 3 
[0]	validation_0-mlogloss:1.59152
[100]	validation_0-mlogloss:1.37433
[121]	validation_0-mlogloss:1.37464
Score Fold 3: 1.0904

 TREINANDO FOLD 4 
[0]	validation_0-mlogloss:1.51281
[100]	validation_0-mlogloss:1.15877
[200]	validation_0-mlogloss:1.15349
[253]	validation_0-mlogloss:1.15684
Score Fold 4: 0.9658


CV SCORE GLOBAL (Média dos Folds): 1.0334
Submissão gerada com média de 5 modelos!


In [12]:
# 1. Gerar predições para o conjunto de teste
print("Gerando predições para o conjunto de teste...")
test_preds = model.predict_proba(X_test.astype('float32'))

# 2. Criar o DataFrame de submissão
submission = pd.DataFrame({'eeg_id': test_df['eeg_id'].values})

# Adicionando as colunas de votos nas posições corretas
target_cols = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
submission[target_cols] = test_preds

# 3. Salvar o arquivo CSV
submission.to_csv('submission.csv', index=False)

display(submission.head())

Gerando predições para o conjunto de teste...


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,173978828,0.024992,0.039428,0.065304,0.019643,0.015114,0.835519
1,942828264,0.010313,0.010322,0.001020,0.020149,0.030929,0.927268
2,1960183550,0.011096,0.009271,0.001476,0.026749,0.030599,0.920809
3,2949584585,0.013213,0.915837,0.003747,0.013733,0.004746,0.048724
4,1040654873,0.576049,0.265794,0.019332,0.032688,0.020697,0.085440
